In [1]:
import os
import sys

sys.path.append(
    os.path.abspath("..")
)

In [2]:
from src.data_loader import load_all_data
from src.preprocessing import *

occupancy, duration, info, adj, distance = load_all_data()

In [3]:
occ_long = convert_wide_to_long(
    occupancy,
    "occupancy"
)

dur_long = convert_wide_to_long(
    duration,
    "duration"
)

In [4]:
master = merge_occupancy_duration(
    occ_long,
    dur_long
)

In [5]:
master.head()

,timestamp,station_id,occupancy,duration
0,1,102,12,0.49
1,2,102,12,0.75
2,3,102,12,0.75
3,4,102,12,0.75
4,5,102,12,0.75


In [6]:
print(info.columns.tolist())
print(info.head())

['num', 'grid', 'count', 'fast_count', 'slow_count', 'area', 'lon', 'la', 'CBD', 'dynamic_pricing']
   num  grid  count  fast_count  slow_count  area       lon        la  CBD  \
0    1   102     30           3          27  0.71  114.1030  22.54041    0   
1    3   105     93           0          93  0.77  114.1208  22.55127    0   
2    5   107     88           2          86  0.89  114.1304  22.54486    0   
3    6   108     39           0          39  1.26  114.1315  22.55403    0   
4    7   109     39           0          39  3.15  114.1430  22.56388    0   

   dynamic_pricing  
0                0  
1                0  
2                0  
3                0  
4                0  


In [7]:
print(occupancy.shape)
print(occupancy.head())

(8640, 248)
   timestamp  102  105  107  108  109  110  111  115  123  ...  1160  1162  \
0          1   12   16   24   15    6    8   24    1    2  ...     0    12   
1          2   12   16   24   15    6    8   24    1    2  ...     0    12   
2          3   12   16   24   15    6    8   24    1    2  ...     0    12   
3          4   12   16   24   15    6    8   24    1    2  ...     0    12   
4          5   12   16   24   15    6    8   24    1    2  ...     0    12   

   1163  1164  1166  1167  1168  1170  1172  1173  
0     1    38    26   162    10     1     8    15  
1     1    38    26   162    10     1     8    15  
2     1    38    26   164    10     1     8    15  
3     1    38    26   166    10     1     8    15  
4     1    38    26   168    10     1     8    15  

[5 rows x 248 columns]


In [8]:
occ_long = occupancy.melt(
    id_vars=occupancy.columns[0],
    var_name="grid",
    value_name="occupancy"
)

occ_long.rename(
    columns={occupancy.columns[0]:"timestamp"},
    inplace=True
)

In [9]:
dur_long = duration.melt(
    id_vars=duration.columns[0],
    var_name="grid",
    value_name="duration"
)

dur_long.rename(
    columns={duration.columns[0]:"timestamp"},
    inplace=True
)

In [10]:
master = occ_long.merge(
    dur_long,
    on=["timestamp","grid"]
)

In [11]:
master.head()

,timestamp,grid,occupancy,duration
0,1,102,12,0.49
1,2,102,12,0.75
2,3,102,12,0.75
3,4,102,12,0.75
4,5,102,12,0.75


In [12]:
info = info.rename(
    columns={"grid":"grid"}
)

In [13]:
master["grid"] = master["grid"].astype(int)

info["grid"] = info["grid"].astype(int)

master = master.merge(
    info,
    on="grid",
    how="left"
)
master.head()

,timestamp,grid,occupancy,duration,num,count,fast_count,slow_count,area,lon,la,CBD,dynamic_pricing
0,1,102,12,0.49,1,30,3,27,0.71,114.103,22.54041,0,0
1,2,102,12,0.75,1,30,3,27,0.71,114.103,22.54041,0,0
2,3,102,12,0.75,1,30,3,27,0.71,114.103,22.54041,0,0
3,4,102,12,0.75,1,30,3,27,0.71,114.103,22.54041,0,0
4,5,102,12,0.75,1,30,3,27,0.71,114.103,22.54041,0,0


# Time Features

In [14]:
# Keep original numeric index
master["time_step"] = master["timestamp"].astype(int)

# Create actual datetime
master["timestamp"] = (
    pd.Timestamp("2024-01-01")
    + pd.to_timedelta(master["time_step"] * 5, unit="min")
)

In [15]:
print(master["timestamp"].head())

0   2024-01-01 00:05:00
1   2024-01-01 00:10:00
2   2024-01-01 00:15:00
3   2024-01-01 00:20:00
4   2024-01-01 00:25:00
Name: timestamp, dtype: datetime64[ns]


In [16]:
master["hour"] = master["timestamp"].dt.hour

master["dayofweek"] = (
    master["timestamp"].dt.dayofweek
)

master["month"] = (
    master["timestamp"].dt.month
)

master["weekend"] = (
    master["dayofweek"] >= 5
).astype(int)

In [17]:
print(master["hour"].nunique())

print(sorted(master["hour"].unique()))

24
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


# Utilisation Features

In [19]:
master["utilization"] = (
    master["duration"] /
    (master["occupancy"] + 1e-6)
)

# Lag Features

In [20]:
master = master.sort_values(
    ["grid","timestamp"]
)

In [21]:
master["occ_lag_1"] = (
    master.groupby("grid")
    ["occupancy"]
    .shift(1)
)

master["occ_lag_12"] = (
    master.groupby("grid")
    ["occupancy"]
    .shift(12)
)

master["occ_lag_24"] = (
    master.groupby("grid")
    ["occupancy"]
    .shift(24)
)

In [22]:
master["rolling_mean_12"] = (
    master.groupby("grid")
    ["occupancy"]
    .transform(
        lambda x:
        x.rolling(12).mean()
    )
)

master["rolling_mean_24"] = (
    master.groupby("grid")
    ["occupancy"]
    .transform(
        lambda x:
        x.rolling(24).mean()
    )
)

In [24]:
print(master.shape)

master.head()

master.isnull().sum().sort_values(
    ascending=False
).head(20)

(2134080, 24)


occ_lag_24         5928
rolling_mean_24    5681
occ_lag_12         2964
rolling_mean_12    2717
occ_lag_1           247
grid                  0
utilization           0
weekend               0
month                 0
dayofweek             0
hour                  0
time_step             0
timestamp             0
CBD                   0
la                    0
lon                   0
area                  0
slow_count            0
fast_count            0
count                 0
dtype: int64

In [25]:
master = master.dropna()

In [26]:
master.head()

,timestamp,grid,occupancy,duration,num,count,fast_count,slow_count,area,lon,...,hour,dayofweek,month,weekend,utilization,occ_lag_1,occ_lag_12,occ_lag_24,rolling_mean_12,rolling_mean_24
24,2024-01-01 02:05:00,102,12,0.75,1,30,3,27,0.71,114.103,...,2,0,1,0,0.0625,12.0,12.0,12.0,12.0,12.0
25,2024-01-01 02:10:00,102,12,0.75,1,30,3,27,0.71,114.103,...,2,0,1,0,0.0625,12.0,12.0,12.0,12.0,12.0
26,2024-01-01 02:15:00,102,12,0.75,1,30,3,27,0.71,114.103,...,2,0,1,0,0.0625,12.0,12.0,12.0,12.0,12.0
27,2024-01-01 02:20:00,102,12,0.75,1,30,3,27,0.71,114.103,...,2,0,1,0,0.0625,12.0,12.0,12.0,12.0,12.0
28,2024-01-01 02:25:00,102,12,0.75,1,30,3,27,0.71,114.103,...,2,0,1,0,0.0625,12.0,12.0,12.0,12.0,12.0


In [27]:
master.to_csv("../data/master_dataset.csv",index=False)